# 05 — Gold Layer and RFM Feature Engineering
## Customer Analytics Platform

Purpose: Join silver tables, calculate RFM scores per customer, and segment customers into High, Medium and Low value groups. This becomes the foundation for the CLV model.

In [0]:
# load silver delta tables for gold layer processing

silver_path = "/Volumes/workspace/default/olist_raw_data/silver"
gold_path   = "/Volumes/workspace/default/olist_raw_data/gold"

customers_df   = spark.read.format("delta").load(f"{silver_path}/customers")
orders_df      = spark.read.format("delta").load(f"{silver_path}/orders")
order_items_df = spark.read.format("delta").load(f"{silver_path}/order_items")
payments_df    = spark.read.format("delta").load(f"{silver_path}/payments")
reviews_df     = spark.read.format("delta").load(f"{silver_path}/reviews")

print("Silver tables loaded for gold processing")

In [0]:
# join orders with payments to get revenue per order

orders_payments = orders_df \
    .join(payments_df, "order_id", "left") \
    .select(
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "delivery_days",
        "is_late_delivery",
        "total_payment_value"
    )

print(f"orders with payments : {orders_payments.count()}")
orders_payments.show(3)

In [0]:
# join with customers to get customer_unique_id
# we need unique_id because same customer can have multiple customer_ids

orders_customers = orders_payments \
    .join(customers_df.select(
        "customer_id", 
        "customer_unique_id",
        "customer_state"
    ), "customer_id", "left")

print(f"orders with customers : {orders_customers.count()}")
orders_customers.show(3)

In [0]:
# calculate RFM features per customer
# R = recency (days since last purchase)
# F = frequency (number of orders)
# M = monetary (total amount spent)

from pyspark.sql.functions import (
    max as spark_max,
    count, 
    sum as spark_sum,
    round as spark_round,
    datediff,
    lit,
    to_date
)

# use last order date in dataset as reference date
reference_date = lit("2018-10-18")

rfm_df = orders_customers \
    .groupBy("customer_unique_id") \
    .agg(
        datediff(
            to_date(reference_date),
            spark_max("order_purchase_timestamp")
        ).alias("recency"),
        count("order_id").alias("frequency"),
        spark_round(
            spark_sum("total_payment_value"), 2
        ).alias("monetary")
    )

print(f"total customers with RFM : {rfm_df.count()}")
print("\nsample RFM scores:")
rfm_df.show(5)

In [0]:
# score each RFM dimension 1-5 using percentile buckets
# 5 = best, 1 = worst

from pyspark.sql.functions import ntile, col
from pyspark.sql.window import Window

# recency: lower is better (bought recently = high value)
# frequency and monetary: higher is better

r_window = Window.orderBy(rfm_df["recency"].desc())
f_window = Window.orderBy(rfm_df["frequency"].asc())
m_window = Window.orderBy(rfm_df["monetary"].asc())

rfm_scored = rfm_df \
    .withColumn("r_score", ntile(5).over(r_window)) \
    .withColumn("f_score", ntile(5).over(f_window)) \
    .withColumn("m_score", ntile(5).over(m_window)) \
    .withColumn("rfm_total_score",
                col("r_score") + 
                col("f_score") + 
                col("m_score"))

print("RFM scores calculated")
print("\nsample scored customers:")
rfm_scored.select(
    "customer_unique_id",
    "recency", "frequency", "monetary",
    "r_score", "f_score", "m_score",
    "rfm_total_score"
).show(5)

In [0]:
# fill null monetary values with 0 and add customer segment labels

from pyspark.sql.functions import when

rfm_scored = rfm_scored \
    .fillna({"monetary": 0.0})

# segment based on rfm_total_score
# score range is 3-15
# high   = 11-15
# medium = 7-10
# low    = 3-6

rfm_segmented = rfm_scored \
    .withColumn("customer_segment",
                when(col("rfm_total_score") >= 11, "High")
                .when(col("rfm_total_score") >= 7,  "Medium")
                .otherwise("Low"))

print("customer segment distribution:")
print("-" * 40)
rfm_segmented.groupBy("customer_segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
# add binary target variable for CLV model
# high value = 1, medium/low = 0

rfm_final = rfm_segmented \
    .withColumn("is_high_value",
                when(col("customer_segment") == "High", 1)
                .otherwise(0))

print("target variable distribution:")
print("-" * 40)
rfm_final.groupBy("is_high_value") \
    .count() \
    .orderBy("is_high_value") \
    .show()

print(f"total customers : {rfm_final.count()}")

In [0]:
# save gold layer rfm table as delta

rfm_final.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/rfm_features")

# verify
gold_df = spark.read.format("delta") \
    .load(f"{gold_path}/rfm_features")

print(f"gold rfm table saved : {gold_df.count()} rows")
print("\nschema:")
gold_df.printSchema()

## Gold Layer Summary

RFM Feature Engineering Complete:
- Total customers analyzed : 93,256
- Reference date used      : 2018-10-18

Customer Segments:
- High value   : 33,703 customers (36%)
- Medium value : 37,051 customers (40%)
- Low value    : 22,502 customers (24%)

Target Variable:
- is_high_value = 1 : High value customers
- is_high_value = 0 : Medium and Low value

Next Step:
- Train XGBoost CLV classification model
- Predict which customers will become high value
- Track experiments with MLflow